In [23]:
train_set = '../data/raw/sample_23.csv' # do not touch
test_set = '../data/raw/sample_24.csv' # do not touch

predictions_weight = {
    '../data/predictions/sklearn_forecast_2024_cluster_specific_combined_case5_clusters.csv': 0.9,
    '../data/predictions/flo/pred_2024_cluster_xgb(6).csv': 0.1,
    # '../data/predictions/flo_static_shapelets/pred_2024_cluster_xgb(7).csv': 0.05,
}

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error


def _resolve_path(path_like):
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (Path.cwd() / path).resolve()


def _pick_id_col(dfs):
    preferred = ["ID", "id", "Id", "series_id", "item_id"]
    common_cols = set(dfs[0].columns)
    for df in dfs[1:]:
        common_cols &= set(df.columns)
    for col in preferred:
        if col in common_cols:
            return col
    for col in dfs[0].columns:
        if col in common_cols:
            return col
    raise ValueError("No shared identifier column found across the prediction files.")


def _pick_value_cols(dfs, id_col):
    common_cols = [c for c in dfs[0].columns if c != id_col]
    for df in dfs[1:]:
        common_cols = [c for c in common_cols if c in df.columns and c != id_col]
    if not common_cols:
        raise ValueError("No shared value columns found across the prediction files.")
    return common_cols


prediction_items = list(predictions_weight.items())
if not prediction_items:
    raise ValueError("predictions_weight must contain at least one prediction file.")

prediction_frames = []
for path, weight in prediction_items:
    df_pred = pd.read_csv(_resolve_path(path))
    prediction_frames.append((path, weight, df_pred))

dfs_only = [df for _, _, df in prediction_frames]
id_col = _pick_id_col(dfs_only)
value_cols = _pick_value_cols(dfs_only, id_col)

common_ids = set(dfs_only[0][id_col].dropna())
for df in dfs_only[1:]:
    common_ids &= set(df[id_col].dropna())
if not common_ids:
    raise ValueError("No overlapping IDs found across the prediction files.")
common_ids = sorted(common_ids)

weighted_sum = None
total_weight = 0.0

for path, weight, df_pred in prediction_frames:
    frame = df_pred[df_pred[id_col].isin(common_ids)].copy()
    frame = frame[[id_col] + value_cols].set_index(id_col)
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame = frame.loc[common_ids]
    if weighted_sum is None:
        weighted_sum = frame * weight
    else:
        weighted_sum = weighted_sum.add(frame * weight, fill_value=0.0)
    total_weight += weight

if total_weight == 0:
    raise ValueError("Total ensemble weight is zero.")

blended_values = weighted_sum / total_weight
blended = blended_values.reset_index()

output_path = _resolve_path("../data/predictions/weighted_average_predictions.csv")
blended.to_csv(output_path, index=False)

test_path = _resolve_path(test_set)
df_true = pd.read_csv(test_path)
if id_col not in df_true.columns:
    raise ValueError(f"Label file does not contain the identifier column '{id_col}'.")

label_value_cols = [c for c in value_cols if c in df_true.columns]
if not label_value_cols:
    raise ValueError("No overlapping value columns found between labels and blended predictions.")

merged = df_true[[id_col] + label_value_cols].merge(
    blended[[id_col] + label_value_cols],
    on=id_col,
    how="inner",
    suffixes=("_true", "_pred"),
)
if merged.empty:
    raise ValueError("No overlapping IDs between the labels and blended predictions.")

y_true = merged[[f"{c}_true" for c in label_value_cols]].to_numpy().ravel()
y_pred = merged[[f"{c}_pred" for c in label_value_cols]].to_numpy().ravel()
mae = mean_absolute_error(y_true, y_pred)

print(f"Saved: {output_path}")
print(f"MAE: {mae:.6f}")
display(blended.head())

Saved: /home/gopes/Documents/Clustering-And-Forecasting-TimeSeries-PlayingGround/data/predictions/weighted_average_predictions.csv
MAE: 3.425779


,ID,2024-01-01,2024-01-02,2024-01-03,2024-01-04,2024-01-05,2024-01-06,2024-01-07,2024-01-08,2024-01-09,...,2024-12-22,2024-12-23,2024-12-24,2024-12-25,2024-12-26,2024-12-27,2024-12-28,2024-12-29,2024-12-30,2024-12-31
0,22,10.004353,9.954395,9.943945,10.114163,9.494003,10.129817,10.420682,9.959523,10.261545,...,9.928294,9.607820,9.576885,9.500543,9.422361,9.504797,9.645757,9.796401,9.718562,9.729368
1,42,43.321812,39.152926,38.000126,38.826942,38.819802,40.307580,41.042156,41.413760,41.031612,...,21.372221,19.093371,19.597407,20.357018,20.485067,20.603696,20.614701,21.227918,19.590509,20.360134
2,56,6.880207,6.662794,6.070022,6.551441,7.364675,8.144236,7.903332,7.897131,7.484620,...,6.385285,5.791003,5.881498,5.918172,5.796382,5.846983,5.906434,5.984787,5.655854,5.755844
3,58,14.958980,15.857177,15.386175,12.903273,13.796371,14.579885,14.503750,13.715808,14.389234,...,12.352322,10.763965,11.062369,11.298597,11.191580,11.302811,11.535421,12.086569,10.914229,11.162206
4,64,2.337374,2.557838,2.480001,2.470379,2.514053,2.625846,2.651115,2.392476,2.547798,...,2.531591,2.427527,2.462451,2.473513,2.476343,2.441936,2.434971,2.492858,2.470466,2.480028
